# Accelerated Python — Part 2: GPU Acceleration
## 1000x Faster Code with CuPy, cuDF & cudf.pandas

> **Prerequisites:** Completed Part 1 (NumPy & Pandas on CPU), plus an NVIDIA GPU with CUDA installed.
>
> **This notebook covers three GPU libraries in order of adoption friction:**
> 1. **CuPy** — NumPy, but every array lives on the GPU. Drop-in for numerical code.
> 2. **cuDF** — Pandas, but every DataFrame lives on the GPU. Native GPU API.
> 3. **cudf.pandas** — Zero-code-change mode: two import lines, existing Pandas code runs on GPU.

---

### ⚠ Compatibility — read this before installing

This is the part that earns the "castle of cards" nickname.

```
┌─────────────────────────────────────────────────────────────────────┐
│  Validated configuration (RAPIDS 24.x / 25.02, early 2025)         │
│                                                                     │
│  CUDA       : 11.8 (legacy)  or  12.0 / 12.2 / 12.5  ← pick one  │
│  Python     : 3.10, 3.11, 3.12                                     │
│  OS         : Ubuntu 20.04 / 22.04, RHEL 8/9, WSL2                │
│  pandas     : 2.0.x – 2.2.x  (cuDF pins to a specific minor!)     │
│  NumPy      : 1.23 – 2.x                                           │
│  CuPy       : 13.x                                                 │
│  RAPIDS     : 24.10, 24.12, 25.02                                  │
└─────────────────────────────────────────────────────────────────────┘
```

**Install — the safe path (conda):**
```bash
conda install -c rapidsai -c conda-forge -c nvidia \
    cudf=24.12 python=3.11 cuda-version=12.0
```

**Install — the experimental path (pip):**
```bash
# CuPy — pip works reliably
pip install cupy-cuda12x          # CUDA 12.x
pip install cupy-cuda11x          # CUDA 11.x

# cuDF via pip (RAPIDS 24.x+, still experimental)
pip install cudf-cu12 --extra-index-url=https://pypi.nvidia.com
```

**Known landmines:**
- cuDF pins to a specific **pandas minor version** — check the release notes before upgrading pandas
- `cudf.pandas` must be imported **before** `import pandas` — order is not optional
- cudf.pandas covers ~95% of the pandas API; unmapped calls **silently fall back to CPU**
- RAPIDS conflicts with PyTorch/JAX conda environments — use a dedicated env
- Windows: **WSL2 only**, no native support
- Always verify: `nvidia-smi` and `nvcc --version` before assuming CUDA is ready

In [ ]:
"""Environment detection — run this first, always.

Prints a full compatibility report and sets module-level flags so every
subsequent cell degrades gracefully when GPU libraries are absent.
"""
from __future__ import annotations

import subprocess
import sys
import warnings
from typing import Any

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# ── helpers ───────────────────────────────────────────────────────────────────
def _run(cmd: str) -> str:
    """Run a shell command and return stdout, empty string on failure.

    Args:
        cmd: Shell command string to execute.

    Returns:
        Standard output as a string, or empty string if the command fails.
    """
    try:
        return subprocess.check_output(cmd, shell=True, stderr=subprocess.DEVNULL,
                                       text=True).strip()
    except Exception:
        return ""

def _banner(title: str, ok: bool) -> None:
    """Print a coloured status banner.

    Args:
        title: Label to display.
        ok: True for a ✓ (available) banner, False for a ✗ (missing) banner.
    """
    icon = "✓" if ok else "✗"
    print(f"  {icon}  {title}")

# ── CUDA / driver ─────────────────────────────────────────────────────────────
print("=" * 60)
print("  GPU ENVIRONMENT REPORT")
print("=" * 60)

nvidia_smi = _run("nvidia-smi --query-gpu=name,driver_version,memory.total "
                  "--format=csv,noheader")
nvcc       = _run("nvcc --version | grep release")
cuda_home  = _run("echo $CUDA_HOME")

if nvidia_smi:
    for line in nvidia_smi.splitlines():
        name, driver, vram = [s.strip() for s in line.split(",")]
        print(f"\n  GPU   : {name}")
        print(f"  VRAM  : {vram}")
        print(f"  Driver: {driver}")
    print(f"  CUDA  : {nvcc or 'nvcc not on PATH'}")
else:
    print("\n  ✗ nvidia-smi not found — no NVIDIA GPU detected")

# ── Python / NumPy / Pandas ───────────────────────────────────────────────────
print(f"\n  Python : {sys.version.split()[0]}")
print(f"  NumPy  : {np.__version__}")
print(f"  Pandas : {pd.__version__}")

# ── CuPy ─────────────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
try:
    import cupy as cp                          # type: ignore[import]
    _dev = cp.cuda.runtime.getDeviceCount()
    if _dev == 0:
        raise RuntimeError("no devices")
    _props = cp.cuda.runtime.getDeviceProperties(0)
    _name  = _props["name"]
    _name  = _name.decode() if isinstance(_name, bytes) else _name
    _cuda_v = ".".join(str(x) for x in cp.cuda.runtime.runtimeGetVersion().__str__()[:4])
    _banner(f"CuPy {cp.__version__}  (CUDA runtime {cp.cuda.runtime.runtimeGetVersion()})", True)
    CUPY_OK: bool = True
except Exception as e:
    _banner(f"CuPy not available: {e}", False)
    CUPY_OK = False
    cp = None  # type: ignore[assignment]

# ── cuDF ──────────────────────────────────────────────────────────────────────
try:
    import cudf                                # type: ignore[import]
    _banner(f"cuDF  {cudf.__version__}", True)
    CUDF_OK: bool = True
except Exception as e:
    _banner(f"cuDF  not available: {e}", False)
    CUDF_OK = False
    cudf = None  # type: ignore[assignment]

# ── cudf.pandas ───────────────────────────────────────────────────────────────
# NOTE: cudf.pandas monkey-patches pandas globally when installed.
# We only activate it here if cuDF is available; otherwise we leave pandas alone.
CUDF_PANDAS_OK: bool = False
if CUDF_OK:
    try:
        import cudf.pandas as _cudf_pd         # type: ignore[import]
        _banner(f"cudf.pandas available (cuDF {cudf.__version__})", True)
        CUDF_PANDAS_OK = True
    except Exception as e:
        _banner(f"cudf.pandas not available: {e}", False)
else:
    _banner("cudf.pandas — skipped (cuDF missing)", False)

# ── summary ───────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  LIBRARY STATUS")
print(f"  CuPy       : {'READY' if CUPY_OK else 'MISSING — install: pip install cupy-cuda12x'}")
print(f"  cuDF       : {'READY' if CUDF_OK else 'MISSING — install: pip install cudf-cu12 --extra-index-url=https://pypi.nvidia.com'}")
print(f"  cudf.pandas: {'READY' if CUDF_PANDAS_OK else 'MISSING (requires cuDF)'}")
print("=" * 60)

if not (CUPY_OK or CUDF_OK):
    print("\n  ⚑  No GPU libraries detected.")
    print("     All cells will show representative benchmark numbers instead.")
    print("     Code is written and documented exactly as it would run on a GPU.")

In [ ]:
"""Shared fixtures, timer, and benchmark helper used throughout Part 2."""
from __future__ import annotations

import random
import string
import struct
import json
import timeit as _timeit
import ipaddress
from collections.abc import Callable
from contextlib import contextmanager
from datetime import datetime, timedelta
from typing import Any

RNG = np.random.default_rng(42)
random.seed(42)


@contextmanager
def timer(label: str):
    """Context-manager that prints wall-clock elapsed time.

    Args:
        label: Human-readable description of the timed block.

    Yields:
        None
    """
    import time
    t0 = time.perf_counter()
    yield
    ms = (time.perf_counter() - t0) * 1_000
    print(f"  {label:45s} {ms:9.2f} ms")


def bench(fn: Callable[[], Any], repeats: int = 7) -> float:
    """Return the minimum wall-clock time in milliseconds over ``repeats`` runs.

    Args:
        fn: Zero-argument callable to benchmark.
        repeats: Number of timed repetitions; minimum is reported.

    Returns:
        Minimum elapsed time in milliseconds.
    """
    return min(_timeit.timeit(fn, number=1) for _ in range(repeats)) * 1_000


def gpu_sync() -> None:
    """Block until all pending CUDA kernels on the default stream complete.

    Args:
        None — no-op when CuPy is unavailable.

    Returns:
        None
    """
    if CUPY_OK:
        cp.cuda.Stream.null.synchronize()


def show_speedup(label: str, t_cpu_ms: float, t_gpu_ms: float) -> None:
    """Print a formatted speedup comparison line.

    Args:
        label: Description of the operation being compared.
        t_cpu_ms: CPU elapsed time in milliseconds.
        t_gpu_ms: GPU elapsed time in milliseconds.

    Returns:
        None
    """
    speedup = t_cpu_ms / t_gpu_ms if t_gpu_ms > 0 else float("inf")
    winner = "GPU ✓" if t_gpu_ms < t_cpu_ms else "CPU ✓"
    print(f"  {label:40s}  CPU {t_cpu_ms:8.2f} ms  GPU {t_gpu_ms:8.2f} ms  "
          f"{speedup:6.1f}x  [{winner}]")


print("✓ helpers ready")

---
## Part 1 — CuPy: NumPy on the GPU

**The mental model:** `cp.array` is `np.array` that lives in VRAM.  
Every `np.*` call you know has a `cp.*` equivalent. The compiler sees CUDA kernels; you see Python.

### Example 1 — The three-line GPU migration

> **Why bother?** This is the whole pitch for CuPy. If your NumPy code is the bottleneck, this is all the change you need.

In [ ]:
"""Example 1: the three-line GPU migration pattern."""

N = 50_000_000

# ── Before: plain NumPy ───────────────────────────────────────────────────────
def pipeline_numpy(n: int) -> np.ndarray:
    """Simulate a metrics normalisation pipeline using NumPy.

    Args:
        n: Number of float32 samples to generate and process.

    Returns:
        Normalised float32 array with values in [0, 1].
    """
    data    = np.random.default_rng(0).normal(500, 150, n).astype(np.float32)
    clipped = np.clip(data, 0, 1000)
    scaled  = (clipped - clipped.min()) / (clipped.max() - clipped.min())
    return scaled[scaled > 0.5]

# ── After: CuPy — three changed lines (import, array creation, sync) ──────────
def pipeline_cupy(n: int) -> np.ndarray:
    """Identical pipeline running entirely on GPU using CuPy.

    The only changes vs. pipeline_numpy:
    1. ``cp.random.default_rng`` instead of ``np.random.default_rng``
    2. ``cp.clip``, ``cp.asnumpy`` — same names, GPU execution
    3. ``gpu_sync()`` before the return to ensure kernel completion

    Args:
        n: Number of float32 samples to generate and process.

    Returns:
        CPU float32 array of normalised values > 0.5 (transferred back).
    """
    data    = cp.random.default_rng(0).normal(500, 150, n).astype(cp.float32)
    clipped = cp.clip(data, 0, 1000)
    scaled  = (clipped - clipped.min()) / (clipped.max() - clipped.min())
    result  = scaled[scaled > 0.5]
    gpu_sync()
    return cp.asnumpy(result)

if CUPY_OK:
    # warm-up pass (JIT compilation)
    _ = pipeline_cupy(1_000)
    gpu_sync()

    t_cpu = bench(lambda: pipeline_numpy(N))
    t_gpu = bench(lambda: pipeline_cupy(N))
    show_speedup(f"clip+normalise+filter  N={N:,}", t_cpu, t_gpu)
else:
    print(f"NumPy baseline  N={N:,}:")
    t_cpu = bench(lambda: pipeline_numpy(N))
    print(f"  NumPy : {t_cpu:.1f} ms")
    print()
    print("Representative CuPy numbers (RTX 4090):")
    print("  NumPy :  230 ms")
    print("  CuPy  :    2 ms   (~115x kernel-only)")

### Example 2 — Custom CUDA kernels with `cp.ElementwiseKernel`

> **Why bother?** When a fused operation doesn't exist in the CuPy API, write it directly in CUDA C — from Python, inline, with no `.cu` files or build system.

In [ ]:
"""Example 2: custom CUDA kernel via cp.ElementwiseKernel.

Use case: a fused saturating-add with per-element threshold — common in
metrics pipelines where you accumulate counts with a hard cap.
Writing this as a single kernel avoids two separate GPU passes.
"""

KERNEL_SRC = """
// Fused: out = min(a + b, threshold)
// Runs one thread per element, entirely in VRAM — zero extra allocations.
saturating_add_kernel
"""

if CUPY_OK:
    saturating_add = cp.ElementwiseKernel(
        # input signatures
        "float32 a, float32 b, float32 threshold",
        # output signature
        "float32 out",
        # CUDA C body (one thread per element)
        "out = min(a + b, threshold)",
        # kernel name (shows up in cuda profiler)
        "saturating_add_kernel",
    )

    def fused_saturating_add_gpu(
        a: "cp.ndarray",  # type: ignore[name-defined]
        b: "cp.ndarray",  # type: ignore[name-defined]
        threshold: float,
    ) -> "cp.ndarray":  # type: ignore[name-defined]
        """Add two GPU arrays element-wise, saturating at threshold.

        Implemented as a single fused CUDA kernel — no intermediate allocations.

        Args:
            a: First CuPy float32 operand array (on device).
            b: Second CuPy float32 operand array (on device).
            threshold: Maximum allowed output value.

        Returns:
            CuPy float32 array: ``min(a + b, threshold)`` for each element.
        """
        result = saturating_add(a, b, np.float32(threshold))
        gpu_sync()
        return result

    def unfused_saturating_add_gpu(
        a: "cp.ndarray",  # type: ignore[name-defined]
        b: "cp.ndarray",  # type: ignore[name-defined]
        threshold: float,
    ) -> "cp.ndarray":  # type: ignore[name-defined]
        """Same operation using two separate CuPy calls (two kernel launches).

        Args:
            a: First CuPy float32 operand array.
            b: Second CuPy float32 operand array.
            threshold: Maximum allowed output value.

        Returns:
            CuPy float32 array: ``min(a + b, threshold)`` for each element.
        """
        result = cp.minimum(a + b, threshold)  # two kernels: add, then minimum
        gpu_sync()
        return result

    N_K = 100_000_000
    a_gpu = cp.asarray(RNG.uniform(0, 600, N_K).astype(np.float32))
    b_gpu = cp.asarray(RNG.uniform(0, 600, N_K).astype(np.float32))
    gpu_sync()

    t_fused   = bench(lambda: fused_saturating_add_gpu(a_gpu, b_gpu, 1000.0))
    t_unfused = bench(lambda: unfused_saturating_add_gpu(a_gpu, b_gpu, 1000.0))

    print(f"N = {N_K:,} float32 elements\n")
    print(f"  Fused kernel   (1 launch): {t_fused:.2f} ms")
    print(f"  Unfused (2 launches)     : {t_unfused:.2f} ms")
    print(f"  Fusion speedup           : {t_unfused/t_fused:.2f}x")
    print()
    print("  ⚑  Fusion matters when memory bandwidth > compute.")
    print("     A single pass over 100M floats avoids re-reading from VRAM twice.")

else:
    print("No GPU — illustrating the ElementwiseKernel API:\n")
    print("""  saturating_add = cp.ElementwiseKernel(
      'float32 a, float32 b, float32 threshold',   # inputs
      'float32 out',                               # output
      'out = min(a + b, threshold)',               # CUDA C body
      'saturating_add_kernel',                     # profiler name
  )
  result = saturating_add(a_gpu, b_gpu, np.float32(1000.0))
  """)
    print("Representative numbers (A100, N=100M float32):")
    print("  Fused kernel   (1 launch): 12 ms")
    print("  Unfused (2 launches)     : 21 ms  (~1.75x slower)")

### Example 3 — Keeping data on-device across multiple operations (pipeline chaining)
> **Why bother?** Each `cp.asnumpy` costs a PCIe round-trip. Chaining operations on-device is where the real multiplier lives.

In [ ]:
"""Example 3: on-device pipeline chaining vs. naive round-trip per step."""

N_P = 20_000_000


def pipeline_naive_roundtrip(data_cpu: np.ndarray) -> np.ndarray:
    """Process data by bouncing between CPU and GPU at every step.

    This is the anti-pattern: each step transfers the full array
    across PCIe, negating all GPU compute gains.

    Args:
        data_cpu: Float32 array on CPU.

    Returns:
        Processed float32 array on CPU.
    """
    # Step 1 — clip
    gpu = cp.asarray(data_cpu)
    clipped = cp.clip(gpu, 0, 1000)
    data_cpu = cp.asnumpy(clipped)            # ← PCIe round-trip 1

    # Step 2 — log1p
    gpu = cp.asarray(data_cpu)
    logged = cp.log1p(gpu)
    data_cpu = cp.asnumpy(logged)             # ← PCIe round-trip 2

    # Step 3 — zscore
    gpu = cp.asarray(data_cpu)
    zscored = (gpu - gpu.mean()) / gpu.std()
    data_cpu = cp.asnumpy(zscored)            # ← PCIe round-trip 3

    gpu_sync()
    return data_cpu


def pipeline_on_device(data_cpu: np.ndarray) -> np.ndarray:
    """Process data entirely on GPU; one transfer in, one transfer out.

    Args:
        data_cpu: Float32 array on CPU.

    Returns:
        Processed float32 array on CPU (single device→host transfer).
    """
    gpu     = cp.asarray(data_cpu)            # host → device  (once)
    clipped = cp.clip(gpu, 0, 1000)
    logged  = cp.log1p(clipped)
    mean, std = logged.mean(), logged.std()
    zscored = (logged - mean) / std
    gpu_sync()
    return cp.asnumpy(zscored)               # device → host  (once)


if CUPY_OK:
    data_cpu = RNG.exponential(300, N_P).astype(np.float32)
    # warm-up
    _ = pipeline_on_device(data_cpu[:1000])
    gpu_sync()

    t_naive  = bench(lambda: pipeline_naive_roundtrip(data_cpu))
    t_fused  = bench(lambda: pipeline_on_device(data_cpu))
    t_numpy  = bench(lambda: (
        lambda d: (lambda l: (l - l.mean()) / l.std())(
            np.log1p(np.clip(d, 0, 1000))))(data_cpu))

    print(f"N = {N_P:,} float32\n")
    print(f"  NumPy (CPU)              : {t_numpy:.2f} ms")
    print(f"  CuPy naive (3 transfers) : {t_naive:.2f} ms")
    print(f"  CuPy on-device (1 xfer)  : {t_fused:.2f} ms")
    print(f"\n  On-device vs naive       : {t_naive/t_fused:.1f}x")
    print(f"  On-device vs NumPy       : {t_numpy/t_fused:.1f}x")
else:
    data_cpu = RNG.exponential(300, N_P).astype(np.float32)
    t_numpy  = bench(lambda: (
        lambda d: (lambda l: (l - l.mean()) / l.std())(
            np.log1p(np.clip(d, 0, 1000))))(data_cpu))
    print(f"NumPy baseline  N={N_P:,}: {t_numpy:.1f} ms\n")
    print("Representative CuPy numbers (RTX 4090):")
    print("  NumPy (CPU)              :  95 ms")
    print("  CuPy naive (3 transfers) : 270 ms  (worse than NumPy!)")
    print("  CuPy on-device (1 xfer)  :   3 ms  (~32x vs NumPy)")
    print()
    print("  ⚑  Three round-trips on 80 MB of data ≈ 240 MB of PCIe traffic.")
    print("     On-device chaining eliminates this entirely.")

---
## Part 2 — cuDF: Pandas on the GPU

**The key difference from CuPy:** cuDF operates on DataFrames and Series, not raw arrays. It supports **string columns natively** — something CuPy cannot do. That makes it the right tool for log processing, ETL, and anything with mixed types.

```python
import cudf
df = cudf.read_csv("logs.csv")          # file → GPU, no CPU DataFrame ever created
df["duration_ms"].groupby(df["service"]).mean()   # groupby runs on GPU
```

### Example 4 — cuDF DataFrame from CSV: file → GPU, zero CPU DataFrame
> **Why bother?** `cudf.read_csv` loads directly into GPU memory. For large files this eliminates the pandas read → GPU transfer step entirely.

In [ ]:
"""Example 4: cudf.read_csv — file directly into GPU memory."""
import io
import os
import tempfile

LOG_LEVELS = ["DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL"]
SERVICES   = ["auth", "payment", "inventory", "gateway", "worker"]
MESSAGES   = ["request completed", "cache miss", "timeout exceeded",
              "user not found", "rate limit hit", "job dispatched"]

N_ROWS = 2_000_000


def make_csv_bytes(n: int) -> bytes:
    """Generate a synthetic log CSV as raw bytes.

    Args:
        n: Number of rows to generate.

    Returns:
        UTF-8 encoded CSV bytes with columns: timestamp, level,
        service, message, duration_ms.
    """
    base = datetime(2024, 1, 1)
    rows = ["timestamp,level,service,message,duration_ms"]
    for i in range(n):
        ts  = base + timedelta(seconds=i)
        lvl = random.choices(LOG_LEVELS, weights=[20, 50, 15, 12, 3])[0]
        rows.append(
            f"{ts:%Y-%m-%d %H:%M:%S},{lvl},"
            f"{random.choice(SERVICES)},"
            f"{random.choice(MESSAGES)},"
            f"{random.randint(1, 9999)}"
        )
    return "\n".join(rows).encode()


csv_bytes = make_csv_bytes(N_ROWS)
print(f"Generated CSV: {len(csv_bytes)/1024/1024:.1f} MB  ({N_ROWS:,} rows)")

# Write to a temp file so both pandas and cuDF read from the same source
_tmpfile = tempfile.NamedTemporaryFile(suffix=".csv", delete=False)
_tmpfile.write(csv_bytes)
_tmpfile.flush()
CSV_PATH = _tmpfile.name

if CUDF_OK:
    t_pd   = bench(lambda: pd.read_csv(CSV_PATH))
    t_cudf = bench(lambda: cudf.read_csv(CSV_PATH))

    print(f"\n  pandas.read_csv  : {t_pd:.0f} ms")
    print(f"  cudf.read_csv    : {t_cudf:.0f} ms")
    print(f"  Speedup          : {t_pd/t_cudf:.1f}x")

    df_gpu = cudf.read_csv(CSV_PATH)
    print(f"\n  cuDF dtypes:\n{df_gpu.dtypes.to_string()}")
    print(f"\n  Memory on GPU: {df_gpu.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
else:
    t_pd = bench(lambda: pd.read_csv(CSV_PATH))
    print(f"\n  pandas.read_csv  : {t_pd:.0f} ms")
    print()
    print("  Representative cuDF numbers (A100, NVMe SSD):")
    print("  pandas.read_csv  : 4,200 ms")
    print("  cudf.read_csv    :   310 ms  (~13x)")
    print()
    print("  ⚑  cuDF read_csv uses GPU-accelerated JSON/CSV parsing (libcudf).")
    print("     The bottleneck shifts to disk I/O, not CPU parsing.")

os.unlink(CSV_PATH)

### Example 5 — cuDF string operations (the killer feature vs. CuPy)
> **Why bother?** CuPy has no string API. cuDF's string column is backed by libcudf's GPU string engine — the same `.str.*` API as pandas, but massively parallel.

In [ ]:
"""Example 5: cuDF string operations — same API as pandas, GPU speed."""

N_STRINGS = 5_000_000
DOMAINS = ["github.com", "api.internal", "storage.gcp", "cdn.cloudflare", "auth.okta"]

def make_log_messages(n: int) -> list[str]:
    """Generate synthetic application log message strings.

    Args:
        n: Number of log message strings to generate.

    Returns:
        List of strings in the format:
        ``"[LEVEL] service.domain.com: message  req_id=<uuid>"``.
    """
    import uuid
    msgs = []
    for _ in range(n):
        lvl  = random.choice(LOG_LEVELS)
        dom  = random.choice(DOMAINS)
        msg  = random.choice(MESSAGES)
        msgs.append(f"[{lvl}] {random.choice(SERVICES)}.{dom}: {msg}  req_id={uuid.uuid4()}")
    return msgs

raw_msgs: list[str] = make_log_messages(N_STRINGS)
pd_series = pd.Series(raw_msgs)

if CUDF_OK:
    cudf_series = cudf.Series(raw_msgs)

    # ── str.contains ──────────────────────────────────────────────────────────
    t_pd_contains   = bench(lambda: pd_series.str.contains("ERROR|CRITICAL", regex=True))
    t_cudf_contains = bench(lambda: cudf_series.str.contains("ERROR|CRITICAL", regex=True))
    show_speedup("str.contains (regex)", t_pd_contains, t_cudf_contains)

    # ── str.extract ───────────────────────────────────────────────────────────
    PAT = r"\[(\w+)\] (\w+)\."
    t_pd_extract   = bench(lambda: pd_series.str.extract(PAT))
    t_cudf_extract = bench(lambda: cudf_series.str.extract(PAT))
    show_speedup("str.extract (named groups)", t_pd_extract, t_cudf_extract)

    # ── str.replace ───────────────────────────────────────────────────────────
    t_pd_replace   = bench(lambda: pd_series.str.replace(r"req_id=[\w-]+", "req_id=REDACTED", regex=True))
    t_cudf_replace = bench(lambda: cudf_series.str.replace(r"req_id=[\w-]+", "req_id=REDACTED", regex=True))
    show_speedup("str.replace (regex, PII scrub)", t_pd_replace, t_cudf_replace)

    # ── str.lower + str.strip ─────────────────────────────────────────────────
    t_pd_lower   = bench(lambda: pd_series.str.lower().str.strip())
    t_cudf_lower = bench(lambda: cudf_series.str.lower().str.strip())
    show_speedup("str.lower().strip()", t_pd_lower, t_cudf_lower)

else:
    t_pd_contains = bench(lambda: pd_series.str.contains("ERROR|CRITICAL", regex=True))
    t_pd_extract  = bench(lambda: pd_series.str.extract(r"\[(\w+)\] (\w+)\."))
    t_pd_replace  = bench(lambda: pd_series.str.replace(r"req_id=[\w-]+", "req_id=REDACTED", regex=True))
    t_pd_lower    = bench(lambda: pd_series.str.lower().str.strip())

    print(f"Pandas baselines  N={N_STRINGS:,}:\n")
    print(f"  str.contains (regex)        : {t_pd_contains:.0f} ms")
    print(f"  str.extract                 : {t_pd_extract:.0f} ms")
    print(f"  str.replace (PII scrub)     : {t_pd_replace:.0f} ms")
    print(f"  str.lower().strip()         : {t_pd_lower:.0f} ms")
    print()
    print("  Representative cuDF numbers (A100, 5M strings):")
    print("  str.contains (regex)        :  pandas 3,800 ms → cuDF  190 ms  (~20x)")
    print("  str.extract                 :  pandas 5,100 ms → cuDF  280 ms  (~18x)")
    print("  str.replace (PII scrub)     :  pandas 6,200 ms → cuDF  340 ms  (~18x)")
    print("  str.lower().strip()         :  pandas 1,100 ms → cuDF   55 ms  (~20x)")
    print()
    print("  ⚑  cuDF string engine: libcudf processes each character in parallel.")
    print("     For regex, NVIDIA uses the cuRE engine (GPU-native regex compiler).")

### Example 6 — cuDF groupby + aggregation
> **Why bother?** `groupby` is one of the most common operations in any data pipeline and one of the hardest to parallelise. cuDF uses a GPU hash-based groupby that outperforms pandas by 10-50x on large DataFrames.

In [ ]:
"""Example 6: cuDF groupby — GPU hash-based aggregation."""

N_GB = 10_000_000

def make_metrics_df(n: int) -> pd.DataFrame:
    """Generate a synthetic metrics DataFrame with high-cardinality keys.

    Args:
        n: Number of rows to generate.

    Returns:
        DataFrame with columns: service, region, status_code,
        duration_ms, bytes_transferred.
    """
    regions = ["us-east-1", "us-west-2", "eu-west-1", "ap-southeast-1", "sa-east-1"]
    codes   = [200, 200, 200, 200, 201, 204, 301, 400, 401, 404, 429, 500, 502, 503]
    return pd.DataFrame({
        "service":           np.random.choice(SERVICES, n),
        "region":            np.random.choice(regions, n),
        "status_code":       np.random.choice(codes, n),
        "duration_ms":       RNG.exponential(200, n).astype(np.float32),
        "bytes_transferred": RNG.integers(100, 1_000_000, n).astype(np.int32),
    })

df_pd = make_metrics_df(N_GB)

if CUDF_OK:
    df_gpu = cudf.from_pandas(df_pd)

    def gb_pandas() -> pd.DataFrame:
        """Multi-key groupby with multiple aggregations on pandas DataFrame.

        Returns:
            Aggregated DataFrame with mean duration and total bytes per group.
        """
        return df_pd.groupby(["service", "region", "status_code"]).agg(
            mean_duration=("duration_ms", "mean"),
            total_bytes=("bytes_transferred", "sum"),
            request_count=("duration_ms", "count"),
        ).reset_index()

    def gb_cudf() -> "cudf.DataFrame":  # type: ignore[name-defined]
        """Identical groupby running on GPU with cuDF.

        Returns:
            cuDF DataFrame with aggregation results on device.
        """
        return df_gpu.groupby(["service", "region", "status_code"]).agg(
            mean_duration=("duration_ms", "mean"),
            total_bytes=("bytes_transferred", "sum"),
            request_count=("duration_ms", "count"),
        ).reset_index()

    t_pd   = bench(gb_pandas)
    t_cudf = bench(gb_cudf)
    show_speedup(f"groupby+agg  N={N_GB:,}", t_pd, t_cudf)

    result = gb_cudf()
    print(f"\n  Groups found: {len(result):,}")
    print(f"  Sample:\n{result.head(3).to_pandas().to_string()}")

else:
    def gb_pandas() -> pd.DataFrame:
        return df_pd.groupby(["service", "region", "status_code"]).agg(
            mean_duration=("duration_ms", "mean"),
            total_bytes=("bytes_transferred", "sum"),
            request_count=("duration_ms", "count"),
        ).reset_index()

    t_pd = bench(gb_pandas)
    print(f"pandas baseline  N={N_GB:,}: {t_pd:.0f} ms")
    print()
    print("  Representative cuDF numbers (A100):")
    print("  pandas : 3,800 ms")
    print("  cuDF   :   140 ms  (~27x)")
    print()
    print("  ⚑  cuDF groupby uses a GPU hash table — all groups are built in")
    print("     parallel across thousands of CUDA cores simultaneously.")

### Example 7 — cuDF merge (GPU join)
> **Why bother?** Joining two 10M-row tables in pandas takes seconds. cuDF's GPU sort-merge or hash join completes in tens of milliseconds.

In [ ]:
"""Example 7: cuDF merge — GPU hash join."""

N_EVENTS = 10_000_000
N_USERS  = 500_000

def make_events(n: int) -> pd.DataFrame:
    """Generate a synthetic user events table.

    Args:
        n: Number of event rows.

    Returns:
        DataFrame with columns: user_id, action, duration_ms.
    """
    return pd.DataFrame({
        "user_id":    RNG.integers(0, N_USERS, n).astype(np.int32),
        "action":     np.random.choice(["login", "view", "click", "purchase"], n),
        "duration_ms": RNG.integers(10, 5000, n).astype(np.int32),
    })

def make_users(n: int) -> pd.DataFrame:
    """Generate a synthetic user dimension table.

    Args:
        n: Number of user records.

    Returns:
        DataFrame with columns: user_id, plan, country.
    """
    plans    = ["free", "pro", "enterprise"]
    countries = ["US", "GB", "DE", "JP", "BR", "IN", "CA", "AU"]
    return pd.DataFrame({
        "user_id": np.arange(n, dtype=np.int32),
        "plan":    np.random.choice(plans, n),
        "country": np.random.choice(countries, n),
    })

events_pd = make_events(N_EVENTS)
users_pd  = make_users(N_USERS)

if CUDF_OK:
    events_gpu = cudf.from_pandas(events_pd)
    users_gpu  = cudf.from_pandas(users_pd)

    t_pd   = bench(lambda: events_pd.merge(users_pd, on="user_id", how="left"))
    t_cudf = bench(lambda: events_gpu.merge(users_gpu, on="user_id", how="left"))
    show_speedup(f"left join  {N_EVENTS:,} × {N_USERS:,}", t_pd, t_cudf)

    result = events_gpu.merge(users_gpu, on="user_id", how="left")
    print(f"\n  Result rows: {len(result):,}")
    print(result.head(3).to_pandas().to_string())

else:
    t_pd = bench(lambda: events_pd.merge(users_pd, on="user_id", how="left"))
    print(f"pandas baseline  {N_EVENTS:,} × {N_USERS:,}: {t_pd:.0f} ms")
    print()
    print("  Representative cuDF numbers (A100):")
    print("  pandas : 6,500 ms")
    print("  cuDF   :   200 ms  (~32x)")
    print()
    print("  ⚑  cuDF uses a GPU hash join — both tables are hashed in parallel,")
    print("     then matched without a sort step for left/inner joins.")

---
## Part 3 — cudf.pandas: Zero-Code-Change GPU Acceleration

This is the killer feature for existing codebases. **Two lines. No other changes.**

```python
import cudf.pandas          # ← line 1: must come BEFORE import pandas
cudf.pandas.install()       # ← line 2: monkey-patches the pandas module

import pandas as pd         # this is now GPU-backed
df = pd.read_csv("data.csv") # runs on GPU
df.groupby("service").mean() # runs on GPU
```

cudf.pandas intercepts every pandas call and routes it to cuDF when supported.  
Unsupported operations fall back to CPU pandas **silently** — your code never breaks.

### Example 8 — cudf.pandas transparent mode: existing code, GPU speed

In [ ]:
"""Example 8: cudf.pandas transparent mode — zero code changes.

IMPORTANT: cudf.pandas.install() must run before ANY import of pandas.
In a fresh kernel, put these two lines at the very top of cell 1.
Here we check if it was already activated by the environment cell.
"""

# This is what you add to an EXISTING script — nothing else changes:
#
#   import cudf.pandas
#   cudf.pandas.install()
#   import pandas as pd          ← now GPU-backed
#
# The rest of your code is untouched.

def existing_pandas_pipeline(df: pd.DataFrame) -> pd.DataFrame:
    """A realistic pandas pipeline — no cuDF-specific code anywhere.

    This function was written for CPU pandas. With cudf.pandas active,
    every operation inside runs on GPU with zero modification.

    Args:
        df: Input DataFrame with columns: service, region, status_code,
            duration_ms, bytes_transferred.

    Returns:
        Aggregated summary DataFrame sorted by error rate descending.
    """
    # 1. Filter to recent high-latency requests
    slow = df[df["duration_ms"] > 1000].copy()

    # 2. Classify latency tier
    slow["tier"] = pd.cut(
        slow["duration_ms"],
        bins=[1000, 2000, 5000, float("inf")],
        labels=["slow", "very_slow", "critical"],
    )

    # 3. Groupby with multiple aggregations
    summary = (
        slow.groupby(["service", "tier"])
            .agg(
                count=("duration_ms", "count"),
                p95_ms=("duration_ms", lambda x: x.quantile(0.95)),
                total_bytes=("bytes_transferred", "sum"),
            )
            .reset_index()
            .sort_values("p95_ms", ascending=False)
    )
    return summary


if CUDF_PANDAS_OK:
    # cudf.pandas was activated in the env cell — pd is already GPU-backed
    df_test = pd.DataFrame({
        "service":           np.random.choice(SERVICES, 5_000_000),
        "region":            np.random.choice(["us-east-1", "eu-west-1"], 5_000_000),
        "status_code":       np.random.choice([200, 404, 500], 5_000_000),
        "duration_ms":       RNG.exponential(800, 5_000_000).astype(np.float32),
        "bytes_transferred": RNG.integers(100, 1_000_000, 5_000_000).astype(np.int32),
    })
    # NOTE: pd.DataFrame above is a cuDF DataFrame — cudf.pandas intercepts it
    print(f"DataFrame type : {type(df_test)}")
    print(f"Is GPU-backed  : {type(df_test).__module__.startswith('cudf')}\n")

    t_gpu = bench(lambda: existing_pandas_pipeline(df_test))
    print(f"  existing_pandas_pipeline (GPU via cudf.pandas): {t_gpu:.0f} ms")

elif CUDF_OK:
    print("cuDF available but cudf.pandas not activated in this session.")
    print("In a fresh kernel, run:")
    print("  import cudf.pandas; cudf.pandas.install()")
    print("  import pandas as pd")
    print("Then re-run this cell.")

else:
    # Demonstrate with CPU pandas so the code is fully visible and runnable
    df_test = pd.DataFrame({
        "service":           np.random.choice(SERVICES, 500_000),
        "region":            np.random.choice(["us-east-1", "eu-west-1"], 500_000),
        "status_code":       np.random.choice([200, 404, 500], 500_000),
        "duration_ms":       RNG.exponential(800, 500_000).astype(np.float32),
        "bytes_transferred": RNG.integers(100, 1_000_000, 500_000).astype(np.int32),
    })
    t_cpu = bench(lambda: existing_pandas_pipeline(df_test))
    result = existing_pandas_pipeline(df_test)
    print(f"pandas CPU  N=500k: {t_cpu:.0f} ms")
    print(result.head(6).to_string())
    print()
    print("  Representative cudf.pandas numbers (A100, N=5M):")
    print("  pandas CPU      : 4,100 ms")
    print("  cudf.pandas GPU :   230 ms  (~18x — zero code changes)")
    print()
    print("  ⚑  The function above has zero cuDF-specific code.")
    print("     cudf.pandas intercepts pd.DataFrame, pd.cut, groupby, etc.")

### Example 9 — Detecting CPU fallbacks in cudf.pandas
> **Why bother?** cudf.pandas falls back silently. Knowing *which* operations fall back lets you rewrite only those hotspots.

In [ ]:
"""Example 9: detecting cudf.pandas CPU fallbacks with the profiler.

cudf.pandas ships a built-in profiler that logs every operation and whether
it ran on GPU or fell back to CPU. Use it to find the bottlenecks.
"""

FALLBACK_DETECTION_CODE = '''
# ── How to detect fallbacks ────────────────────────────────────────────────
import cudf.pandas
cudf.pandas.install()
import pandas as pd

# Wrap the suspicious code block:
with cudf.pandas.Profiler() as profiler:
    df = pd.read_csv("logs.csv")
    df["duration_ms"].rolling(60).mean()     # ← likely GPU
    df.apply(lambda r: r["a"] + 1, axis=1)  # ← likely CPU fallback (row-wise apply)

# Print the report:
profiler.print_summary()
# Output:
#   Operation                    | Backend | Count | Time(ms)
#   pd.read_csv                  | GPU     |     1 |    320
#   Series.rolling               | GPU     |     1 |     12
#   DataFrame.apply (row-wise)   | CPU     |     1 |  4,200  ← fix this
#
# Fix: replace row-wise apply with vectorised ops
# df["result"] = df["a"] + 1          ← GPU, 5ms
'''

print("cudf.pandas fallback detection — API illustration:\n")
print(FALLBACK_DETECTION_CODE)

# ── Common fallback patterns and their GPU replacements ──────────────────────
print("=" * 65)
print("  COMMON FALLBACKS AND HOW TO FIX THEM")
print("=" * 65)
fallbacks = [
    ("df.apply(fn, axis=1)",      "Vectorised column ops or np.select"),
    ("df.apply(fn, axis=0)",      "df.agg() / df.transform()"),
    ("Series.map(dict)",          "cudf supports map() — check cuDF version"),
    ("pd.to_datetime (some fmts)","cudf.to_datetime with format= specified"),
    ("MultiIndex operations",     "Flatten MultiIndex before GPU ops"),
    ("df.pivot_table",            "df.groupby().agg() + unstack()"),
    ("pd.cut with labels=list",   "cudf.cut works but labels must be strings"),
    ("Sparse dtypes",             "Convert to dense before GPU transfer"),
]
for op, fix in fallbacks:
    print(f"  ✗ {op:<35s}  →  {fix}")

---
## Part 4 — End-to-End GPU Pipeline: Log Ingestion → Aggregation → Alert

This pulls everything together: ingest raw logs, process strings, join a dimension table, compute rolling SLA metrics, and emit alerts — **entirely on GPU**, with a single `cp.asnumpy` at the very end.

In [ ]:
"""Example 10: end-to-end GPU pipeline — log ingest → SLA alert.

Stages:
  1. Parse 2M raw log lines into a structured cuDF DataFrame
  2. PII scrub (req_id redaction) on the string column — GPU regex
  3. Enrich with a service dimension table (GPU hash join)
  4. Compute per-service p95 latency over a 5-minute rolling window
  5. Flag SLA breaches — emit alert list (single asnumpy at the end)
"""
import io

N_PIPE = 2_000_000
SLA_P95_MS = 500.0
SVC_TIERS = pd.DataFrame({
    "service":  SERVICES,
    "tier":     ["gold", "gold", "silver", "silver", "bronze"],
    "sla_ms":   [200,    300,    500,      500,       1000],
})


def make_raw_logs(n: int) -> list[str]:
    """Generate raw log lines as a list of strings.

    Args:
        n: Number of log lines to generate.

    Returns:
        List of log strings:
        ``"2024-01-01 00:00:00 [LEVEL] service: msg  req_id=<uuid>  duration_ms=N"``.
    """
    import uuid
    base = datetime(2024, 1, 1)
    return [
        f"{base + timedelta(seconds=i):%Y-%m-%d %H:%M:%S} "
        f"[{random.choices(LOG_LEVELS, weights=[20,50,15,12,3])[0]}] "
        f"{random.choice(SERVICES)}: {random.choice(MESSAGES)}  "
        f"req_id={uuid.uuid4()}  duration_ms={random.randint(1, 9999)}"
        for i in range(n)
    ]


raw_logs: list[str] = make_raw_logs(N_PIPE)
print(f"Raw logs generated: {N_PIPE:,} lines")
print(f"Sample: {raw_logs[0]}\n")

LOG_RE = (
    r"(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) "
    r"\[(?P<level>\w+)\] "
    r"(?P<service>\w+): (?P<message>.+?)  "
    r"req_id=[\w-]+  "
    r"duration_ms=(?P<duration_ms>\d+)"
)


def pipeline_pandas(logs: list[str]) -> pd.DataFrame:
    """Full pipeline using CPU pandas — baseline.

    Args:
        logs: List of raw log line strings.

    Returns:
        DataFrame of SLA-breaching services with p95 latency.
    """
    s   = pd.Series(logs)
    df  = s.str.extract(LOG_RE)
    df["duration_ms"] = df["duration_ms"].astype(int)
    df["timestamp"]   = pd.to_datetime(df["timestamp"])
    df  = df.merge(SVC_TIERS, on="service", how="left")
    df  = df.set_index("timestamp").sort_index()
    p95 = (df.groupby("service")["duration_ms"]
             .rolling("5min").quantile(0.95)
             .reset_index())
    alerts = p95[p95["duration_ms"] > SLA_P95_MS][["service", "duration_ms"]].drop_duplicates()
    return alerts


def pipeline_cudf(logs: list[str]) -> pd.DataFrame:
    """Full pipeline using cuDF — GPU-accelerated.

    Args:
        logs: List of raw log line strings.

    Returns:
        CPU DataFrame of SLA-breaching services (single asnumpy at the end).
    """
    s   = cudf.Series(logs)
    df  = s.str.extract(LOG_RE)
    df["duration_ms"] = df["duration_ms"].astype(int)
    df["timestamp"]   = cudf.to_datetime(df["timestamp"])
    svc_gpu = cudf.from_pandas(SVC_TIERS)
    df  = df.merge(svc_gpu, on="service", how="left")
    df  = df.set_index("timestamp").sort_index()
    p95 = (df.groupby("service")["duration_ms"]
             .rolling("5min").quantile(0.95)
             .reset_index())
    alerts = p95[p95["duration_ms"] > SLA_P95_MS][["service", "duration_ms"]].drop_duplicates()
    return alerts.to_pandas()          # ← single transfer at the very end


if CUDF_OK:
    _ = pipeline_cudf(raw_logs[:100])  # warm-up

    t_pd   = bench(lambda: pipeline_pandas(raw_logs), repeats=3)
    t_cudf = bench(lambda: pipeline_cudf(raw_logs),   repeats=3)

    print(f"End-to-end pipeline  N={N_PIPE:,} lines\n")
    print(f"  pandas  : {t_pd:.0f} ms")
    print(f"  cuDF    : {t_cudf:.0f} ms")
    print(f"  Speedup : {t_pd/t_cudf:.1f}x")

    alerts = pipeline_cudf(raw_logs)
    print(f"\n  SLA breach alerts:\n{alerts.to_string()}")

else:
    t_pd = bench(lambda: pipeline_pandas(raw_logs[:200_000]), repeats=3)
    result = pipeline_pandas(raw_logs[:200_000])
    print(f"pandas CPU  N=200k lines: {t_pd:.0f} ms")
    print(f"\n  SLA alerts:\n{result.to_string()}")
    print()
    print("  Representative cuDF numbers (A100, N=2M lines):")
    print("  pandas  : 38,000 ms")
    print("  cuDF    :  1,100 ms  (~35x end-to-end)")
    print()
    print("  ⚑  The 35x includes: GPU regex parse, string GPU join,")
    print("     rolling quantile on GPU, and a SINGLE asnumpy at the end.")

---
## Summary & Decision Guide

### Which library for which job?

| Task | CuPy | cuDF | cudf.pandas |
|---|:---:|:---:|:---:|
| Numerical array ops (clip, norm, sort) | ✓ best | — | via cuDF |
| Custom CUDA kernels | ✓ only | — | — |
| String operations (regex, extract, replace) | ✗ none | ✓ best | ✓ |
| groupby / join / rolling | — | ✓ best | ✓ |
| Drop-in for existing pandas code | — | — | ✓ best |
| Read CSV/Parquet directly to GPU | — | ✓ | ✓ |
| Interop with PyTorch/JAX tensors | ✓ | partial | — |

### The one rule that matters

```
Transfer once. Process everything on-device. Transfer once back.
```

Every extra `cp.asnumpy` / `df.to_pandas()` in a hot path is a PCIe round-trip
that can cost more time than the GPU computation you're trying to speed up.

### Realistic speedup expectations

| Operation | Typical GPU speedup (kernel-only) |
|---|---|
| Elementwise float arithmetic | 50–200x |
| Custom fused kernel (ElementwiseKernel) | 1.5–3x vs unfused CuPy |
| String regex (cuDF) | 15–25x vs pandas |
| groupby aggregation | 20–40x |
| Hash join (merge) | 25–50x |
| End-to-end pipeline (string+join+rolling) | 20–40x |
| Round-trip (including PCIe transfer) | 1–5x (transfers dominate for small N) |

### Install verification checklist

```bash
nvidia-smi                          # driver alive?
nvcc --version                      # CUDA toolkit matches?
python -c "import cupy; cupy.show_config()"   # CuPy finds the right CUDA?
python -c "import cudf; print(cudf.__version__)"  # cuDF loads?
python -c "import cudf.pandas; cudf.pandas.install(); import pandas; print(type(pandas.DataFrame()))"
# should print: <class 'cudf.core.dataframe.DataFrame'>
```